## 1. The audit
Aggregating this data was very difficult. But in order to do it, every single Period Transaction Report (will be referred to as PTR) was downloaded to my computer, and ran through Google Gemini 3.5 Flash to become .json files with a consistent and readable format. Of the 1231 Digital PTR Documents, 1229 ran perfectly, and the first and third largest PDF files were too large for Gemini 3.5's context window of ~50,000 tokens. 

At a later date I want to read the 195 handwritten PTR documents using the LLM, and I also want to verify that the output of the LLM was good and sound, but its about time that I got to do some data analysis, and I will be making do with what I have, which is having parsed 86% of the files.

The dataset that will be loaded are the 1229 properly generated .json files, each corresponding to a digital PTR report that was transformed into a readable json via LLM.

In [ ]:
import pandas as pd

base_url = "https://raw.githubusercontent.com/TheCarsonWest/DSA405/refs/heads/master/data/json/"
index_url = "https://raw.githubusercontent.com/TheCarsonWest/DSA405/refs/heads/master/data/all_json_paths.txt"

document_ids = (
    pd.read_csv(index_url, header=None, dtype=str)[0]
    .str.strip()
    .dropna()
)
document_ids = document_ids[document_ids != ""].tolist()

json_files = [f"{document_id}.json" for document_id in document_ids]
dataframes = []

for file_name in json_files:
    df = pd.read_json(f"{base_url}{file_name}")
    df["file_name"] = file_name
    dataframes.append(df)

combined_df = (
    pd.concat(dataframes, ignore_index=True)
    if dataframes
    else pd.DataFrame()
)

print(f"Found {len(json_files)} JSON files.")
print(f"Combined DataFrame shape: {combined_df.shape}")
combined_df.info()

Found 0 JSON files in ./data/json directory.
Combined DataFrame shape: <bound method DataFrame.info of Empty DataFrame
Columns: []
Index: []>


In [58]:
# assert the number of unique file names matches the number of JSON files read
assert combined_df["file_name"].nunique() == len(json_files), "Mismatch in unique file names and JSON files read."


KeyError: 'file_name'

Now to parse the JSON rows into a useful format:

In [52]:
transaction_columns = pd.json_normalize(
    combined_df["transactions"].map(
        lambda value: value if isinstance(value, dict) else {}
    ),
    sep="_"
)

parsed_df = pd.concat(
    [
        combined_df.drop(columns="transactions").reset_index(drop=True),
        transaction_columns.reset_index(drop=True),
    ],
    axis=1
)

print("Parsed DataFrame shape:", parsed_df.shape)


Parsed DataFrame shape: (12880, 12)


Creating a `ticker` column, for attempting to find the ticker for any asset that should have public stock data.

In [53]:
# Making a ticker column by searching for a regex pattern between parentheses in the 'ast' column
parsed_df["ticker"] = parsed_df["ast"].str.extract(r"\((.*?)\)")[0]

# removing values in the 'ticker' column that are longer than 5 characters
parsed_df.loc[parsed_df["ticker"].str.len() > 5, "ticker"] = None
# This removes the values of 128 rows

print(parsed_df.ticker.value_counts(dropna=False))

ticker
NaN      1677
MSFT      251
NVDA      196
AAPL      147
None      128
         ... 
HURC        1
ARINA       1
YOU         1
CHPT        1
BAYZF       1
Name: count, Length: 1573, dtype: int64


# Part 1, the Audit:

In [45]:
print("dtypes of parsed DataFrame:")
print(parsed_df.dtypes)

print("-"*50)

dtypes of parsed DataFrame:
file_name    object
own          object
ast          object
code         object
typ          object
dt           object
ndt          object
amt          object
cg             bool
stat         object
sub          object
desc         object
ticker       object
dtype: object
--------------------------------------------------


| Column Name | Description                                                                                                     | Loaded dtype | Dtype as it should be                   |
| ----------- | --------------------------------------------------------------------------------------------------------------- | ------------ | --------------------------------------- |
| file_name   | name of the file the row came from                                                                              | String       | String                                  |
| own         | owner code of the asset as reported                                                                             | String       | String                                  |
| ast         | Full name of the asset (including ticker in parentheses if it is a stock)                                       | String       | String                                  |
| code        | Type of asset (list of abbreviations can be found [Here](https://fd.house.gov/reference/asset-type-codes.aspx)) | String       | String                                  |
| typ         | transaction type                                                                                                | String       | String                                  |
| dt          | date of transaction                                                                                             | String       | date                                    |
| ndt         | date the congressperson reported that they were notified of the transaction                                     | String       | date                                    |
| amt         | size of transactions in binned amounts (i.e $1,001 - $15,000, $15,001 - $50,000)                                | String       | Int (requires further processing later) |
| cg          | If the transaction resulted in a Capital gains >$200                                                            | Boolean      | Boolean                                 |
| stat        | Filling status of transaction                                                                                   | String       | String                                  |
| sub         | Subholding Account Name                                                                                         | String       | String                                  |
| desc        | Additional description the filer provides                                                                       | String       | String                                  |
| ticker      | attempted capture of a stock ticker                                                                             | String       | String                                  |

In [46]:
# Fixing date columns to datetime format
for column in ["dt", "ndt"]:
    parsed_df[column] = pd.to_datetime(
        parsed_df[column],
        format="%m/%d/%Y",
        errors="coerce"
    )
# Theres apparently a lot of malformed sentinel dates that make no sense, so we will filter them out
lower_bound = pd.Timestamp("2023-10-01")
upper_bound = pd.Timestamp("2026-01-01")

for column in ["dt", "ndt"]:
    outside_range = (
        (parsed_df[column] < lower_bound)
        | (parsed_df[column] > upper_bound)
    )
    parsed_df.loc[outside_range, column] = pd.NaT

print("-"*50)
print(parsed_df['dt'].describe())
print("-"*50)
print(parsed_df['ndt'].describe())
print("-"*50)
# Ranges
print("dt range:", parsed_df['dt'].min(), "to", parsed_df['dt'].max())
print("ndt range:", parsed_df['ndt'].min(), "to", parsed_df['ndt'].max())
# Percent of missing values
print("-"*50)
missing_dt = parsed_df['dt'].isna().mean() * 100
missing_ndt = parsed_df['ndt'].isna().mean() * 100
print(f"Percentage of missing values in 'dt': {missing_dt:.2f}%")
print(f"Percentage of missing values in 'ndt': {missing_ndt:.2f}%")
# Count of missing values
print("-"*50)
print(f"Count of missing values in 'dt': {parsed_df['dt'].isna().sum()}")
print(f"Count of missing values in 'ndt': {parsed_df['ndt'].isna().sum()}")

--------------------------------------------------
count                             9424
mean     2025-01-11 15:46:27.096774144
min                2023-10-02 00:00:00
25%                2024-08-28 18:00:00
50%                2025-03-10 00:00:00
75%                2025-06-03 00:00:00
max                2025-12-30 00:00:00
Name: dt, dtype: object
--------------------------------------------------
count                             9847
mean     2025-01-23 13:30:00.548390400
min                2023-10-02 00:00:00
25%                2024-08-13 00:00:00
50%                2025-03-31 00:00:00
75%                2025-07-29 00:00:00
max                2025-12-30 00:00:00
Name: ndt, dtype: object
--------------------------------------------------
dt range: 2023-10-02 00:00:00 to 2025-12-30 00:00:00
ndt range: 2023-10-02 00:00:00 to 2025-12-30 00:00:00
--------------------------------------------------
Percentage of missing values in 'dt': 26.83%
Percentage of missing values in 'ndt': 23.55%
---

Wow, 3000+ dropped values for each of the two date rows! That's some terrible turnover and it's probably my fault.

---

### Categorical Variables

In [47]:
# Value Counts for categorical columns
categorical_columns = ["typ", "stat", "code"]

for column in categorical_columns:
    print(f"Value counts for column '{column}':")
    print(parsed_df[column].value_counts(dropna=False))
    print("-"*50)

Value counts for column 'typ':
typ
P              6433
S              4139
S (partial)    2248
E                60
Name: count, dtype: int64
--------------------------------------------------
Value counts for column 'stat':
stat
New        12780
Amended       93
Deleted        7
Name: count, dtype: int64
--------------------------------------------------
Value counts for column 'code':
code
ST    10909
GS     1075
OT      339
CS      200
HN      107
OP       91
CT       52
PS       46
OI       27
OL       15
AB       11
ET        5
VA        2
RS        1
Name: count, dtype: int64
--------------------------------------------------


All of these categories look pretty fine to me. 

# 2: Data dictionary:
`dt` and `ndt`

| Field         | What goes in it                                                                                                                  |
| ------------- | -------------------------------------------------------------------------------------------------------------------------------- |
| Variable name | `dt` and `ndt`                                                                                                                   |
| Type          | date                                                                                                                             |
| Units         | Days                                                                                                                             |
| Valid range   | 11/14/2022 - 1/1/2026                                                                                                            |
| Missingness   | `dt` 26.83% of dates were out of range<br>`ndt` 23.55% of dates were out of range                                                |
| Notes         | `dt` is the recorded date of transaction<br>`ndt` is the recorded date the congressperson<br>was notified of the transaction<br> |

`cg`

| Field         | What goes in it                                      |
| ------------- | ---------------------------------------------------- |
| Variable name | `cg`                                                 |
| Type          | Boolean                                              |
| Units         | True or False                                        |
| Missingness   | 0%                                                   |
| Notes         | If the transaction resulted in a Capital gains >$200 |

`code`

| Field         | What goes in it                                                                                                                                              |
| ------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| Variable name | `code`                                                                                                                                                       |
| Type          | Category                                                                                                                                                     |
| Factor levels | ST, GS, OT, CS, HN, OP, CT, PS, OI, OL, AB, ET, VA, RS<br><br>all codes can be found on [this website](https://fd.house.gov/reference/asset-type-codes.aspx) |
| Missingness   | 0%                                                                                                                                                           |
| Notes         | We will mostly be looking at ST and ET, since they have searchable stock tickers                                                                             |

`stat`

| Field         | What goes in it                                            |
| ------------- | ---------------------------------------------------------- |
| Variable name | stat (status)                                              |
| Type          | category                                                   |
| Factor levels | New, Amended, Deleted                                      |
| Missingness   | 0%                                                         |
| Notes         | We will just be looking at  transactions with a New status |

`typ`

| Field         | What goes in it                                                 |
| ------------- | --------------------------------------------------------------- |
| Variable name | typ (type)                                                      |
| Type          | Category                                                        |
| Units         | P, S, S (partial), E                                            |
| Missingness   | 0%                                                              |
| Notes         | Denotes a purchase, sale, partial sale, or exchange of an asset |

`filename`

| Field         | What goes in it                                                                                                                                                                                       |
| ------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Variable name | file_name                                                                                                                                                                                             |
| Type          | String                                                                                                                                                                                                |
| Valid range   | (currently) the integer value of the <br>file name should be greater than 20000000<br><br>once I process the handwritten PTR reports, there will be document ids that are between 8000000 and 9000000 |
| Missingness   | 0%                                                                                                                                                                                                    |
| Notes         | Attached to every row is the json file it came from. We will use this later to append the information about its publish date, and the congressperson who filed it.                                    |


# 3: cleaning log
In this data, we will **not** be removing duplicate values, since it is totally plausible that some congressperson made the exact same asset transaction twice in one day to throw off any would-be watchful eyes. Also, since the data comes straight from an LLM, guided by a well-formed prompt, there is suprisingly little cleaning to do.

| #   | Column   | Change made                                                               | Rows/cells affected | Why                                                                  | What is lost                                                        |
| --- | -------- | ------------------------------------------------------------------------- | ------------------- | -------------------------------------------------------------------- | ------------------------------------------------------------------- |
| 1   | `dt`     | Turned all values out of plausible date range (11/14/22- 1/1/26) to `NaT` | 3456                | The dates should be impossible and are likely malformed/hallucinated | Dates that are likely junk data                                     |
| 2   | `ndt`    | Turned all values out of plausible date range (11/14/22- 1/1/26) to `NaT` | 3033                | The dates should be impossible and are likely malformed/hallucinated | Dates that are likely junk data                                     |
| 3   | `ticker` | Removed captured tickers that had a string length greater than 5          | 128                 | Stock tickers cannot be longer than 5 characters                     | Some sentinels that accidentally became recognized as stock tickers |



In [56]:

print("Final parsed DataFrame shape:", parsed_df.shape)

Final parsed DataFrame shape: (12880, 13)


# 4. Row and column accounting

```txt
- Rows in raw files                     12880
- Rows removed as exact duplicates      0
- Rows removed as near-duplicates       0
- Rows removed for other reasons        0   
- Rows in cleaned file                  12880
```

---

```txt
- Columns in raw file                   2   (raw json string and the file name it came from)
- Columns dropped                       1   (was just a json text block, needed splitting into columns)
- Columns added                         12  (11 were from splitting up the json text block, 1 was extracted from one of those rows)
- Columns in cleaned file               13
```

# 5. The Provenance Brief

This dataset is from the accumulation of Periodic Transaction Reports from 2023-2025 from members of the House of Representitives. The data was originally stored as individual PDF documents who were listed in a directory XML file. I (Carson West), ran a specialized prompt to get a Large Language Model (Google Gemini 3.5 Flash) to transform 1,229 Digital PTR documents into 1,229 JSON files containing all of the information about each individual transaction contained within the Periodic Transaction Reports. The 1,229 processed documents resulted in a total 12,280 recorded transactions. Each individual row contains the Document it came from, the description of the asset, the dates it occured and the congressperson was notified, the various other information about the transaction include the approximate size in dollars, whether it was a buy, sell or puchase, if the transaction resulted in capital gains over $200, and an additional description that the filer may have entered. In processing, I also attempted to find the stock ticker by searching for text within parentheses in the full asset description.